# 🤖 03 — Machine Learning Models

This notebook demonstrates ML analysis of survey data:

- **Classification**: Predict satisfaction groups from survey responses
- **Feature importance**: Identify key drivers using SHAP
- **Hyperparameter tuning**: Optimize model performance
- **Segmentation**: Discover respondent clusters
- **Cluster profiling**: Understand segment characteristics

---

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from survey_toolkit import (
    SurveyCleaner,
    SurveyStats,
    SurveyClassifier,
    SurveySegmentation,
    generate_sample_survey,
    detect_column_types,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_style("whitegrid")

print("✅ Setup complete")

## 2. Prepare Data

In [ ]:
df = generate_sample_survey(
    n_respondents=500,
    n_likert_items=10,
    random_state=42,
)

types = detect_column_types(df)
likert_cols = types["likert"]
demo_cols = types["categorical"]

clean_df = (
    SurveyCleaner(df)
    .remove_speeders("duration_seconds", min_seconds=60)
    .remove_straightliners(likert_cols, threshold=0.95)
    .handle_missing(strategy="median", columns=likert_cols)
    .get_clean_data()
)

print(f"📋 Clean data: {clean_df.shape}")
print(f"📋 Target distribution:")
print(clean_df["satisfaction_group"].value_counts())

---
## Part A: Classification

Predict **satisfaction group** (Dissatisfied / Neutral / Satisfied)
from Likert-scale responses.

### 3. Prepare Features & Target

In [ ]:
classifier = SurveyClassifier(clean_df)
X, y = classifier.prepare_data(
    feature_cols=likert_cols,
    target_col="satisfaction_group",
    scale=True,
)

print(f"📊 Features shape: {X.shape}")
print(f"📊 Target shape: {y.shape}")
print(f"📊 Target classes: {np.unique(y)}")
print(f"📊 Class distribution:")
print(y.value_counts().sort_index())

if classifier.label_encoder:
    print(f"\n📊 Label mapping:")
    for label, code in zip(
        classifier.label_encoder.classes_,
        range(len(classifier.label_encoder.classes_))
    ):
        print(f"  {label} → {code}")

### 4. Model Comparison

Compare multiple classifiers using **stratified k-fold cross-validation**.

In [ ]:
results = classifier.run_model_comparison(cv_folds=5, scoring="accuracy")
print("\n📊 Model Comparison Results:")
display(results)

In [ ]:
# Visualize model comparison
fig, ax = plt.subplots(figsize=(10, 5))

models = results["model"].values
means = results["mean_score"].values
stds = results["std_score"].values
colors = plt.cm.Set2(np.linspace(0, 1, len(models)))

bars = ax.barh(models, means, xerr=stds, color=colors,
               edgecolor="white", capsize=5)
ax.set_xlabel("Accuracy (5-Fold CV)")
ax.set_title("Model Comparison — Satisfaction Prediction")
ax.set_xlim(0, 1)

# Add value labels
for bar, mean, std in zip(bars, means, stds):
    ax.text(mean + std + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{mean:.3f} ± {std:.3f}",
            va="center", fontsize=10)

plt.tight_layout()
plt.savefig("../outputs/figures/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

### 5. Classification Report (Best Model)

In [ ]:
report = classifier.get_classification_report()

print(f"📊 Best Model: {report['model']}")
print(f"\n📊 Classification Report:")

report_df = pd.DataFrame(report["classification_report"]).T
display(report_df.round(4))

In [ ]:
# Confusion Matrix
cm = np.array(report["confusion_matrix"])
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=report["target_names"],
    yticklabels=report["target_names"],
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {report['model']}")
plt.tight_layout()
plt.savefig("../outputs/figures/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

### 6. Feature Importance (SHAP)

Understand **which survey items** are most important
for predicting satisfaction.

In [ ]:
try:
    importance = classifier.feature_importance()

    print("📊 Feature Importance (SHAP values):")
    display(importance)

    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    top_n = min(15, len(importance))
    top = importance.head(top_n)

    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, top_n))[::-1]
    ax.barh(top["feature"][::-1], top["mean_shap_value"][::-1],
            color=colors, edgecolor="white")
    ax.set_xlabel("Mean |SHAP Value|")
    ax.set_title("Feature Importance — Key Drivers of Satisfaction")
    plt.tight_layout()
    plt.savefig("../outputs/figures/feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()

except ImportError:
    print("⚠️ SHAP not installed. Install with: pip install shap")

### 7. Hyperparameter Tuning

In [ ]:
tuning_result = classifier.hyperparameter_tune(
    model_name="Random Forest",
    param_grid={
        "model__n_estimators": [50, 100, 200],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_split": [2, 5, 10],
    },
    cv_folds=5,
)

print("📊 Hyperparameter Tuning Results:")
print(f"  Best score: {tuning_result['best_score']}")
print(f"  Best params: {tuning_result['best_params']}")

In [ ]:
# Show top 10 parameter combinations
print("\n📊 Top 10 Parameter Combinations:")
for i, combo in enumerate(tuning_result["cv_results_summary"][:10]):
    print(f"  {i+1}. Score: {combo['mean_test_score']:.4f} "
          f"(± {combo['std_test_score']:.4f}) — {combo['params']}")

### 8. Make Predictions

In [ ]:
# Predict on new data
sample_data = clean_df[likert_cols].head(10)
predictions = classifier.predict(sample_data)
probabilities = classifier.predict(sample_data, return_proba=True)

pred_df = pd.DataFrame({
    "respondent": clean_df["respondent_id"].head(10).values,
    "predicted": predictions,
    "actual": clean_df["satisfaction_group"].head(10).values,
})

# Add probabilities
if classifier.label_encoder:
    for i, class_name in enumerate(classifier.label_encoder.classes_):
        pred_df[f"prob_{class_name}"] = probabilities[:, i].round(4)

print("📊 Sample Predictions:")
display(pred_df)

---
## Part B: Respondent Segmentation (Clustering)

Discover natural groupings in the survey data using K-Means clustering.

### 9. Prepare Clustering Data

In [ ]:
segmenter = SurveySegmentation(clean_df)
X_cluster = segmenter.prepare_data(likert_cols, scale=True)

print(f"📊 Clustering data: {X_cluster.shape}")

### 10. Find Optimal Number of Clusters

In [ ]:
optimal = segmenter.find_optimal_k(k_range=range(2, 9))

print(f"\n📊 Optimal k: {optimal['optimal_k']}")
print(f"📊 Best silhouette score: {optimal['best_silhouette']}")

In [ ]:
# Visualize elbow + silhouette
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Elbow plot (Inertia)
axes[0].plot(optimal["k_range"], optimal["inertias"], "bo-", linewidth=2)
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")
axes[0].axvline(x=optimal["optimal_k"], color="red", linestyle="--", alpha=0.5)

# Silhouette scores
axes[1].plot(optimal["k_range"], optimal["silhouette_scores"], "go-", linewidth=2)
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Analysis")
axes[1].axvline(x=optimal["optimal_k"], color="red", linestyle="--", alpha=0.5)

# Calinski-Harabasz
axes[2].plot(optimal["k_range"], optimal["calinski_harabasz_scores"], "ro-", linewidth=2)
axes[2].set_xlabel("Number of Clusters (k)")
axes[2].set_ylabel("Calinski-Harabasz Score")
axes[2].set_title("Calinski-Harabasz Index")
axes[2].axvline(x=optimal["optimal_k"], color="red", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("../outputs/figures/optimal_k.png", dpi=150, bbox_inches="tight")
plt.show()

### 11. Fit Clusters & Profile

In [ ]:
n_clusters = optimal["optimal_k"]
profiles = segmenter.fit_clusters(n_clusters=n_clusters)

print(f"📊 {n_clusters} Clusters Found")
print(f"📊 Silhouette Score: {segmenter.results['silhouette']}")
print(f"📊 Cluster Sizes: {segmenter.results['cluster_sizes']}")
print(f"\n📊 Cluster Profiles (Mean Scores):")
display(profiles.round(3))

In [ ]:
# Heatmap of cluster profiles
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    profiles, annot=True, fmt=".2f", cmap="RdYlGn",
    center=3, vmin=1, vmax=5,
    linewidths=0.5, ax=ax,
)
ax.set_title("Cluster Profiles — Mean Likert Scores")
ax.set_ylabel("Cluster")
ax.set_xlabel("Survey Item")
plt.tight_layout()
plt.savefig("../outputs/figures/cluster_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### 12. Visualize Clusters (PCA)

In [ ]:
viz_df = segmenter.visualize_clusters()

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    viz_df["PC1"], viz_df["PC2"],
    c=viz_df["cluster"], cmap="Set2",
    alpha=0.6, s=50, edgecolors="white", linewidth=0.5,
)
ax.set_xlabel(f"PC1 ({segmenter.results['pca_variance'][0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({segmenter.results['pca_variance'][1]:.1%} variance)")
ax.set_title("Respondent Segments (PCA Projection)")
plt.colorbar(scatter, label="Cluster", ax=ax)
plt.tight_layout()
plt.savefig("../outputs/figures/cluster_pca.png", dpi=150, bbox_inches="tight")
plt.show()

### 13. Demographic Profile of Clusters

In [ ]:
demo_profiles = segmenter.profile_clusters_by_demographics(demo_cols)

for demo_col, profile in demo_profiles.items():
    print(f"\n📊 {demo_col} Distribution by Cluster:")
    display((profile * 100).round(1))

In [ ]:
# Visualize demographic breakdown per cluster
for demo_col, profile in demo_profiles.items():
    fig, ax = plt.subplots(figsize=(10, 5))
    (profile * 100).plot(kind="bar", ax=ax, colormap="Set2", edgecolor="white")
    ax.set_title(f"{demo_col} by Cluster")
    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel("Cluster")
    ax.legend(title=demo_col, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"../outputs/figures/cluster_{demo_col}.png", dpi=150, bbox_inches="tight")
    plt.show()

### 14. Cluster Labels (Back to Original Data)

In [ ]:
labels = segmenter.get_cluster_labels()
segmented_df = clean_df.loc[labels.index].copy()
segmented_df["cluster"] = labels

print(f"📊 Segmented data shape: {segmented_df.shape}")
print(f"\n📊 Cluster × Satisfaction:")
display(pd.crosstab(segmented_df["cluster"], segmented_df["satisfaction_group"], normalize="index").round(3) * 100)

### 15. Naming Segments

Based on the cluster profiles, let's assign descriptive names.

In [ ]:
# Name clusters based on mean scores
cluster_names = {}
for cluster_id in profiles.index:
    mean_score = profiles.loc[cluster_id].mean()
    if mean_score >= 4.0:
        cluster_names[cluster_id] = "Enthusiasts"
    elif mean_score >= 3.3:
        cluster_names[cluster_id] = "Moderates"
    elif mean_score >= 2.5:
        cluster_names[cluster_id] = "Ambivalent"
    else:
        cluster_names[cluster_id] = "Critics"

print("📊 Segment Names:")
for cid, name in cluster_names.items():
    size = segmenter.results["cluster_sizes"][cid]
    mean = profiles.loc[cid].mean()
    print(f"  Cluster {cid} → '{name}' (n={size}, mean={mean:.2f})")

segmented_df["segment_name"] = segmented_df["cluster"].map(cluster_names)

## 16. Summary

In [ ]:
print("=" * 60)
print("ML ANALYSIS SUMMARY")
print("=" * 60)
print(f"\n📊 Classification:")
print(f"  Best model: {classifier.best_model_name}")
print(f"  Accuracy: {results.iloc[0]['mean_score']:.4f}")
if "feature_importance" in classifier.results:
    top_feat = classifier.results["feature_importance"].iloc[0]["feature"]
    print(f"  Top driver: {top_feat}")

print(f"\n👥 Segmentation:")
print(f"  Clusters: {n_clusters}")
print(f"  Silhouette: {segmenter.results['silhouette']}")
for cid, name in cluster_names.items():
    print(f"  • {name}: n={segmenter.results['cluster_sizes'][cid]}")